# Cells

This notebook shows the single-cell side of embpy: model-aware preprocessing, cell-state embeddings, optional single-cell foundation models, plotting by cell line, cell-line annotation, and reusable AnnData outputs.

The executable path uses PCA so the tutorial remains fast. The same API calls are shown for heavier single-cell foundation models such as scGPT, Geneformer, UCE, STATE, and Stack; those cells are opt-in because they may download large model weights or require optional packages.


## embpy capability map

All embpy tutorials follow the same package pattern:

1. `BioEmbedder.embed(...)` is the single entry point for genes, proteins, molecules, perturbation images, text, and single cells.
2. Resolvers convert biological identifiers into model-ready inputs, such as gene sequences, protein sequences, SMILES strings, microscopy tensors, or AnnData matrices.
3. The model registry selects the requested embedding backend, from lightweight local baselines to foundation models.
4. Preprocessing utilities in `embpy.pp` prepare inputs when a model needs a specific representation, such as raw counts, log-normalized expression, or morphology canvases.
5. Metadata tools in `embpy.tl` and `embpy.resources` annotate the resulting AnnData with genes, proteins, molecules, perturbations, and cell-line metadata.
6. Plotting and comparison helpers in `embpy.pl` and `embpy.tl` inspect embedding geometry, cluster structure, KNN overlap, similarity, and annotation enrichment.

The important contract is that embeddings are stored in AnnData-friendly locations: row-aligned embeddings in `.obsm`, feature-aligned embeddings in `.varm`, and entity-aligned payloads or provenance in `.uns`. `.X` stays reserved for count/expression-like data or a lightweight placeholder.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from embpy import BioEmbedder, pl, pp, tl
from embpy.models.singlecell_models import singlecell_info
from embpy.resources import CellLineAnnotator

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]


## Inspect the cell model registry

Single-cell models have different requirements. Some consume raw counts, some consume log-normalized expression, some can decode latents back to expression, and Stack exposes an in-context generation path. embpy records those requirements in the single-cell registry and uses them when `preprocessing="auto"` is requested.


In [ ]:
example_models = ["pca", "scgpt", "geneformer_v2_12L", "uce", "state", "stack", "scvi"]
available = set(embedder.list_available_models("single_cell"))

rows = []
for model_name in example_models:
    if model_name not in available:
        continue
    card = singlecell_info(model_name)
    rows.append(
        {
            "model": model_name,
            "description": card.description,
            "input_layer": card.input_layer,
            "preprocessing_auto": card.default_preprocessing,
            "vocab": card.vocab_type,
            "decode": card.supports_decode,
            "generate": card.supports_generation,
        }
    )

model_table = pd.DataFrame(rows)
display(model_table)


## Build a small cell-line AnnData

The count matrix below is intentionally small and synthetic so the notebook can run quickly anywhere. It still has the structure embpy expects from real single-cell RNA-seq data: cells in `.obs`, genes in `.var_names`, raw counts in `.X`, and biological annotations such as `cell_line`, `tissue`, and `treatment` in `.obs`.


In [ ]:
rng = np.random.default_rng(7)

genes = [
    "TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1",
    "EPCAM", "KRT18", "VIM", "MKI67", "CD3D", "IL7R", "LYZ", "S100A8",
    "HBB", "GATA1", "MPO", "NFKB1", "CXCL8", "IFIT1", "ISG15", "ACTB",
]
cell_lines = ["A549", "K562", "Jurkat", "THP-1"]
tissue = {
    "A549": "lung epithelium",
    "K562": "myeloid leukemia",
    "Jurkat": "T cell leukemia",
    "THP-1": "monocyte leukemia",
}
treatments = ["control", "IFNG", "EGF"]

line_effects = {
    "A549": {"EPCAM": 18, "KRT18": 12, "EGFR": 8},
    "K562": {"HBB": 18, "GATA1": 12, "MPO": 5},
    "Jurkat": {"CD3D": 16, "IL7R": 8, "JUN": 5},
    "THP-1": {"LYZ": 18, "S100A8": 10, "NFKB1": 6},
}
treatment_effects = {
    "control": {},
    "IFNG": {"STAT1": 10, "IRF1": 8, "IFIT1": 6, "ISG15": 6},
    "EGF": {"EGFR": 8, "MYC": 5, "JUN": 5},
}

obs_rows = []
count_rows = []
for line in cell_lines:
    for i in range(30):
        treatment = treatments[i % len(treatments)]
        lam = np.full(len(genes), 2.0)
        for gene, effect in line_effects[line].items():
            lam[genes.index(gene)] += effect
        for gene, effect in treatment_effects[treatment].items():
            lam[genes.index(gene)] += effect
        count_rows.append(rng.poisson(lam).astype(np.int32))
        obs_rows.append(
            {
                "cell_line": line,
                "tissue": tissue[line],
                "treatment": treatment,
                "batch": f"batch_{1 + (i % 2)}",
            }
        )

adata = ad.AnnData(
    X=np.vstack(count_rows),
    obs=pd.DataFrame(obs_rows, index=[f"cell_{i:03d}" for i in range(len(obs_rows))]),
    var=pd.DataFrame(index=pd.Index(genes, name="gene_symbol")),
)
adata.var_names_make_unique()

display(adata)
display(adata.obs.head())


## Preprocess when the model needs it

You can either let `BioEmbedder.embed(..., preprocessing="auto")` choose the model-specific preprocessing, or you can prepare the AnnData explicitly with `embpy.pp.preprocess_counts`. For PCA, the registry says `auto` resolves to the standard pipeline: raw counts are preserved, log-normalized expression goes to `.layers["log_normalized"]`, and highly-variable-gene metadata can be recorded in `.var`.


In [ ]:
prepared = pp.preprocess_counts(
    adata,
    pipeline="standard",
    min_genes=0,
    min_cells=0,
    select_hvg=False,
    target_sum=10_000,
    copy=True,
)

print("layers:", list(prepared.layers.keys()))
print("raw-count X min/max:", float(np.min(prepared.X)), float(np.max(prepared.X)))
print("log-normalized layer min/max:", float(np.min(prepared.layers["log_normalized"])), float(np.max(prepared.layers["log_normalized"])))


## Fit a lightweight baseline and prepare for foundation models

PCA is kept here only as a fast encoder-decoder baseline and as a concrete example of model-aware preprocessing. The real model comparison below uses single-cell foundation-model embeddings, not PCA embeddings.

The baseline consumes the explicitly preprocessed data with `preprocessing="none"`, writes one latent matrix to `.obsm`, and keeps provenance in `.uns["embpy_cell_embeddings"]`.


In [ ]:
cell_space = embedder.embed(
    prepared,
    entity_type="cell",
    model="pca",
    output="anndata",
    preprocessing="none",
    n_pca_components=8,
    pca_use_hvg=False,
    key="X_pca_baseline",
)

decoded = embedder.decode_cells(
    cell_space,
    model="pca",
    obsm_key="X_pca_baseline",
    write_layer="X_pca_baseline_reconstructed",
)

print("PCA baseline obsm keys:", [k for k in cell_space.obsm if k.startswith("X_pca")])
print("decoded expression shape:", decoded.shape)
print("reconstruction layer:", "X_pca_baseline_reconstructed" in cell_space.layers)
print("preprocessing metadata:")
display(pd.Series(cell_space.uns["embpy_cell_embeddings"]["__preprocessing__"]))


## Embed cells with foundation models

This is the main cell-embedding step. Each requested foundation model writes one row-aligned matrix to `.obsm`, so the same AnnData can hold several views of the same cells.

`scgpt`, `geneformer_v2_12L`, and `uce` are backed by Helical. In this repository, Helical is intentionally isolated in a Linux-only pixi environment because it pins `torch`, `scipy`, and `transformers` versions that conflict with the normal `dev` and `gpu` environments. Run this section from the cluster with the `Python (embpy-helical)` kernel, not from the default/dev kernel.

Cluster setup:

```bash
cd /lustre/groups/ml01/workspace/goncalo.pinto/embpy
pixi install -e helical-gpu
pixi run -e helical-gpu verify-helical
pixi run -e helical-gpu install-kernel-helical
pixi run -e helical-gpu jupyter --port 8899 --ip 0.0.0.0
```

Then open this notebook and select `Python (embpy-helical)` as the kernel. On a CPU node, use `helical-cpu` instead of `helical-gpu`, but expect the models to be slower.


In [ ]:
try:
    import helical  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "The selected foundation models require Helical, but this kernel does not have `helical` installed. "
        "Run this notebook on Linux with the Helical pixi environment:\n"
        "  pixi install -e helical-gpu\n"
        "  pixi run -e helical-gpu verify-helical\n"
        "  pixi run -e helical-gpu install-kernel-helical\n"
        "Then restart this notebook with the `Python (embpy-helical)` kernel."
    ) from exc

foundation_models = ["scgpt", "geneformer_v2_12L", "uce"]
foundation_errors: dict[str, str] = {}

for model_name in foundation_models:
    key = f"X_{model_name}"
    try:
        cell_space = embedder.embed(
            cell_space,
            entity_type="cell",
            model=model_name,
            output="anndata",
            preprocessing="auto",
            batch_size=8,
            min_genes=0,
            min_cells=0,
            key=key,
        )
    except Exception as exc:
        foundation_errors[model_name] = f"{type(exc).__name__}: {exc}"

foundation_keys = [f"X_{model_name}" for model_name in foundation_models if f"X_{model_name}" in cell_space.obsm]
print("foundation obsm keys:", foundation_keys)

if foundation_errors:
    display(pd.Series(foundation_errors, name="foundation_model_errors"))

if len(foundation_keys) < 2:
    raise RuntimeError(
        "Foundation-model embedding did not produce at least two usable embeddings. "
        "The plot/comparison section needs at least two `.obsm` matrices. "
        "Inspect `foundation_model_errors` above; common causes are missing model weights, "
        "Hugging Face/network access problems, or insufficient GPU memory."
    )

print(f"Ready to compare {len(foundation_keys)} foundation-model embeddings.")


## Plot and compare foundation-model cell embeddings

The plots and diagnostics below compare different single-cell foundation-model views of the same cells. UMAP is used only as a 2-D visualization method applied to the foundation embeddings; the embeddings being compared are `X_scgpt`, `X_geneformer_v2_12L`, `X_uce`, or whichever foundation models you selected above.

The plotting helpers operate directly on AnnData. Here we color a foundation-model embedding by cell-line identity, treatment, tissue, and batch. This is the pattern you would use for real CCLE/DepMap, Lamin, or local AnnData datasets.


In [ ]:
primary_key = foundation_keys[0]
comparison_key = foundation_keys[1]

pl.plot_embedding_space(
    cell_space,
    obsm_key=primary_key,
    method="umap",
    color="cell_line",
    annotate=False,
    title=f"{primary_key} cell embeddings colored by cell line",
)

pl.embedding_color_panel(
    cell_space,
    obsm_key=primary_key,
    method="umap",
    color_keys=["cell_line", "treatment", "tissue", "batch"],
    ncols=2,
    title=f"Same {primary_key} foundation embedding, different annotations",
)

k = min(10, cell_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(cell_space, primary_key, comparison_key, k=k)
print(f"Mean {primary_key}/{comparison_key} KNN overlap: {mean_overlap:.3f}")
pl.knn_overlap(cell_space, obsm_keys=foundation_keys, k=k)
pl.cross_embedding_correlation(cell_space, primary_key, comparison_key)
pl.embedding_norms(cell_space, obsm_keys=foundation_keys)


## Annotate per cell line

`CellLineAnnotator` aggregates public metadata from Cellosaurus, DepMap/CCLE, and Cell Model Passports when network access is available. It writes compact columns to `.obs` and stores full annotation records in `.uns`, so the annotation follows the same AnnData object as the embeddings and plots.


In [ ]:
RUN_CELL_LINE_API_ANNOTATION = True

if RUN_CELL_LINE_API_ANNOTATION:
    annotator = CellLineAnnotator(rate_limit_delay=0.1)
    try:
        cell_space = annotator.annotate_adata(
            cell_space,
            column="cell_line",
            sources=["cellosaurus", "depmap", "passports"],
        )
    except Exception as exc:  # Public APIs can be unavailable in offline notebooks.
        print(f"Cell-line annotation skipped: {type(exc).__name__}: {exc}")
    else:
        display(compact_obs(cell_space, ("cellline_",), base=["cell_line", "tissue", "treatment"]).drop_duplicates("cell_line"))
        print("annotation stores:", [k for k in cell_space.uns if "annotation" in k])


## Summary

In this notebook you saw the full single-cell workflow: inspect model requirements, preprocess an AnnData when needed, fit a lightweight decoder baseline, embed cell states with foundation models, compare foundation-model embedding spaces, plot them by cell-line annotations, and enrich the same AnnData with external cell-line metadata.


In [ ]:
cell_space.write_h5ad(OUTPUT_DIR / "cell_line_embedding_tutorial.h5ad")
print(OUTPUT_DIR / "cell_line_embedding_tutorial.h5ad")
